Viewed featurize.py:48-74

Voici l'explication complète et détaillée, module par module, argument par argument, du **Modèle de référence 1 (k-mer + Régression Logistique)**.

---

## 🎯 Le Concept Global du Modèle 1

Les algorithmes de Machine Learning traditionnels ne savent pas lire du texte brut (comme `"ATGC..."`). 
Le but du Modèle 1 est donc :
1. **Transformer** chaque chaîne d'ADN en un vecteur de 256 chiffres (les fréquences des mots de 4 lettres).
2. **Entraîner** un modèle mathématique linéaire (**Régression Logistique**) sur ces chiffres pour prédire la classe (`1` = codant, `0` = non-codant).
3. **Évaluer** la précision sur un jeu de validation jamais vu pendant l'entraînement.

---

## 🧩 Étape 1 : La Vectorisation (`kmer_matrix`)

### Le Code de la cellule :
```python
from featurize import kmer_matrix

K = 4
X_train_kmer = kmer_matrix(train["sequence"], k=K)
X_val_kmer = kmer_matrix(val["sequence"], k=K)
y_train, y_val = train["label"].to_numpy(), val["label"].to_numpy()
```

### Explication des modules et arguments :

1. **`from featurize import kmer_matrix`**
   - **C'est quoi ?** Un module personnalisé situé dans [src/featurize.py](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day1/src/featurize.py#L49).
   - **Pourquoi l'utiliser ?** Il contient la fonction qui parcourt automatiquement chaque séquence texte d'un tableau et calcule les fréquences des k-mers.

2. **Argument `k = 4` (`K = 4`)**
   - **Pourquoi `4` ?** En génétique, les codons (qui codent les acides aminés) font 3 lettres. Prendre $k=4$ permet d'englober des tri-nucléotides + 1 lettre de contexte ($4^4 = 256$ combinaisons). C'est un compromis idéal : assez riche pour capturer l'information biologique, mais petit et rapide à calculer ($256$ colonnes).

3. **`train["sequence"]` et `val["sequence"]`**
   - **C'est quoi ?** La colonne Pandas contenant la liste de nos 4 000 chaînes d'ADN de 200 pb.
   - **Résultat :** `kmer_matrix` renvoie une matrice NumPy de taille **`(4000, 256)`**.

4. **`train["label"].to_numpy()`**
   - **Pourquoi `.to_numpy()` ?** Convertit la colonne de réponses Pandas (`0` ou `1`) en un tableau NumPy 1D de forme `(4000,)`, le format standard attendu par scikit-learn.

---

## 🤖 Étape 2 : Création et Entraînement du Modèle (`make_kmer_classifier`)

### Le Code de la cellule :
```python
from models.baselines import make_kmer_classifier

kmer_clf = make_kmer_classifier("logreg")
kmer_clf.fit(X_train_kmer, y_train)
```

### Explication des modules et arguments :

1. **`from models.baselines import make_kmer_classifier`**
   - **C'est quoi ?** Une fonction "usine" dans [src/models/baselines.py](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day1/src/models/baselines.py#L18).

2. **Argument `kind = "logreg"`**
   - **Pourquoi "logreg" ?** Demande à l'usine d'instancier une **`LogisticRegression(max_iter=1000)`** de `scikit-learn`.
   - **Pourquoi `max_iter=1000` dans le code sous-jacent ?** La régression logistique cherche les meilleurs poids par un algorithme d'optimisation itératif. Par défaut `max_iter=100` peut s'arrêter trop tôt ; `1000` garantit que l'algorithme a le temps de converger vers la solution optimale.
   - **Pourquoi la Régression Logistique dans ce cas ?** C'est le modèle de référence binaire le plus simple, le plus rapide à entraîner (quelques millisecondes) et le plus interprétable.

3. **Méthode `.fit(X_train_kmer, y_train)`**
   - **Argument 1 (`X_train_kmer`) :** La matrice $4000 \times 256$ de fréquences.
   - **Argument 2 (`y_train`) :** Les 4 000 réponses réelles (`0` ou `1`).
   - **Que fait `.fit()` ?** Il ajuste 256 poids mathématiques ($w_1, w_2, ..., w_{256}$) : si un 4-mer est très fréquent dans les gènes, son poids devient positif ; s'il est fréquent hors des gènes, son poids devient négatif.

---

## 📊 Étape 3 : Évaluation du Modèle (`evaluate_sklearn`)

### Le Code de la cellule :
```python
from eval import evaluate_sklearn

kmer_metrics = evaluate_sklearn(kmer_clf, X_val_kmer, y_val)
print("k-mer + logreg:", kmer_metrics)
```

### Explication des modules et arguments :

1. **`from eval import evaluate_sklearn`**
   - **C'est quoi ?** Notre outil de mesure défini dans [src/eval.py](file:///Users/user/PythonProjects/learning/EEIA-bioAI-Workshop-project/day1/src/eval.py#L38).

2. **Argument 1 : `model = kmer_clf`**
   - Le modèle qu'on vient d'entraîner avec `.fit()`.

3. **Argument 2 : `X = X_val_kmer`**
   - **Pourquoi les données de validation ?** Ce sont 4 000 séquences que le modèle **n'a jamais vues** lors du `.fit()`. Cela permet de vérifier s'il a vraiment "compris" la biologie ou s'il a juste "appris par cœur" (*overfitting*).

4. **Argument 3 : `y = y_val`**
   - Les vraies étiquettes de validation pour vérifier les réponses du modèle.

5. **Ce que fait la fonction en interne :**
   - Elle exécute `preds = model.predict(X_val_kmer)` pour obtenir les devinettes du modèle.
   - Elle calcule et renvoie l'**Accuracy** (taux de succès) et le **F1-Score** (équilibre des classes).

---

### 📝 Résumé de la chaîne complète du Modèle 1

$$\text{Séquences ADN (Texte)} \xrightarrow{\text{kmer\_matrix}} (4000, 256) \xrightarrow{\text{make\_kmer\_classifier + .fit()}} \text{Modèle Entraîné} \xrightarrow{\text{evaluate\_sklearn}} \{\text{accuracy}: 0.85, \text{f1}: 0.85\}$$

Edited classifier_heads.py
Edited classifier_heads.py
Viewed baselines.py:1-31

Bien sûr ! C'est la meilleure approche. Prenons le temps de comprendre **le cours et la théorie** derrière PyTorch et les réseaux de neurones avant de coder quoi que ce soit.

---

# 📚 Le Cours : Du Réseau de Neurones (MLP) à la Boucle d'Entraînement

---

## 1. Pourquoi avons-nous besoin d'une « Tête MLP » ?

Au **Jour 1**, vous avez extrait des caractéristiques à la main (k-mers).
Au **Jour 2**, le modèle de fondation **Evo2** prend une séquence ADN et la transforme en un **vecteur numérique de 4096 dimensions** (qu'on appelle un *embedding*).

Mais **4096 nombres**, ce n'est pas une réponse « Codant » (1) ou « Non-codant » (0).
C'est là qu'intervient la **tête de classification (MLPHead)** : c'est un petit réseau de neurones qu'on ajoute par-dessus pour apprendre à séparer ces 4096 nombres en deux catégories.

```
Séquence ADN ──> [Evo2 (Gelé)] ──> Embedding (4096 dims) ──> [Tête MLP] ──> Logit (Score brut)
```

---

## 2. Anatomie du réseau `MLPHead`

Notre réseau `MLPHead` est composé de **4 blocs successifs** (assemblés avec `nn.Sequential`) :

```
Entrée (4096) ──> [ Linear ] ──> (128) ──> [ ReLU ] ──> [ Dropout ] ──> [ Linear ] ──> Sortie (1 logit)
```

1. **`nn.Linear(d_in, d_hidden)`** *(Couche linéaire / Dense)* :
   - Prend les `d_in` dimensions en entrée (ex: 4096) et les projette sur `d_hidden` neurones (ex: 128).
   - Elle effectue l'opération mathématique : $y = W \cdot x + b$ (multiplication par une matrice de poids $W$ + un biais $b$).

2. **`nn.ReLU()`** *(Fonction d'activation non-linéaire)* :
   - Elle remplace toutes les valeurs négatives par $0$ ($f(x) = \max(0, x)$).
   - **Pourquoi ?** Sans activation non-linéaire, combiner plusieurs couches linéaires équivaudrait mathématiquement à une seule couche linéaire ! ReLU permet au réseau d'apprendre des frontières de décision complexes.

3. **`nn.Dropout(dropout)`** *(Régularisation)* :
   - Pendant l'entraînement, cette couche « éteint » aléatoirement un pourcentage des neurones (ex: 10% si `dropout=0.1`).
   - **Pourquoi ?** Pour empêcher le réseau de devenir trop dépendant de certains neurones spécifiques (évite le surapprentissage / *overfitting*).

4. **`nn.Linear(d_hidden, 1)`** *(Couche de sortie)* :
   - Réduit les 128 neurones à **1 seul nombre** (le score du modèle).

---

## 3. C'est quoi un `logit` et pourquoi `squeeze(-1)` ?

- La sortie du réseau est un **`logit`** : c'est un score réel non-borné (qui peut aller de $-\infty$ à $+\infty$).
  - Si le logit est **très positif** (ex: `+4.2`) $\rightarrow$ Le modèle est très confiant que la séquence est **Codante (1)**.
  - Si le logit est **très négatif** (ex: `-3.1`) $\rightarrow$ Le modèle est très confiant que la séquence est **Non-codante (0)**.
  - Si le logit est proche de `0.0` $\rightarrow$ Le modèle hésite à 50/50.

- **Pourquoi la méthode `forward` fait `return self.net(x).squeeze(-1)` ?**
  - Pour un mini-lot (*batch*) de 256 séquences, la sortie brute de `self.net(x)` a la forme `(256, 1)` (un tableau 2D à 1 colonne).
  - Mais nos vraies étiquettes $y$ ont la forme `(256,)` (un tableau 1D).
  - `.squeeze(-1)` supprime la dernière dimension de taille 1 pour obtenir exactement `(256,)`, afin d'aligner les formes pour le calcul d'erreur.

---

## 4. La Boucle d'Entraînement PyTorch (Les 5 Étapes Majeures)

L'apprentissage d'un réseau se fait en répétant un cycle de **5 étapes** sur chaque mini-batch de données :

```
┌─────────────────────────────────────────────────────────┐
│ 1. Forward ──> 2. Loss ──> 3. Zero_grad ──> 4. Backward ──> 5. Step │
└─────────────────────────────────────────────────────────┘
```

1. **`logits = teacher(X_batch)`** *(Forward)* :
   On passe les données d'entrée dans le réseau pour obtenir les prédictions brutes (`logits`).

2. **`loss = criterion(logits, y_batch)`** *(Calcul de la perte / erreur)* :
   On utilise **`nn.BCEWithLogitsLoss()`** (Binary Cross-Entropy). Elle compare le logit prédit avec la vraie réponse (`0` ou `1`) et calcule une note d'erreur (`loss`).
   - Plus le modèle se trompe, plus la `loss` est élevée.

3. **`optimizer.zero_grad()`** *(Remise à zéro des gradients)* :
   En PyTorch, les gradients s'accumulent par défaut. Avant de calculer les nouveaux gradients de ce mini-batch, on doit effacer les anciens !

4. **`loss.backward()`** *(Rétropropagation)* :
   PyTorch remonte le réseau à l'envers et calcule le gradient (l'impact) de chaque poids $W$ sur l'erreur finale.

5. **`optimizer.step()`** *(Mise à jour des poids)* :
   L'optimiseur (**Adam**) utilise les gradients calculés pour ajuster légèrement les poids $W$ dans la direction qui réduit l'erreur.

---

Prenez le temps de relire ces 4 points. Est-ce que ces concepts (Linear, ReLU, Dropout, Logit, et la boucle à 5 étapes) sont clairs pour vous, ou souhaitez-vous qu'on approfondisse un point en particulier ?